## Trainning The Evaluation Function Weights


In [26]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

### About Data
- data is taken from stockfish static evaluation (depth=0)
- our current evaluation function is static evaluation.
- we havent added check/checkmate evaluation so check positions are removed from dataset to reduce huge fluctuations in Error.

### Representing Data in 2D grid format

In [29]:

EMPTY = 0
# Piece Representations
BLACK_PAWN = -1
BLACK_ROOK = -2
BLACK_KNIGHT = -3
BLACK_BISHOP = -4
BLACK_QUEEN = -5
BLACK_KING = -6

WHITE_PAWN = 1
WHITE_ROOK = 2
WHITE_KNIGHT = 3
WHITE_BISHOP = 4
WHITE_QUEEN = 5
WHITE_KING = 6


def fromFEN(fen):
    # Creates an Empty Board of 8x8
    board = []
    for i in range(8):
        row=[]
        for j in range(8):
            row.append(EMPTY)
        board.append(row)

    row = 0
    col = 0

    for c in fen:
        # if c reaches an empty character then board representation ends
        if c==' ':
            break
        # if / is encountered move to next row
        if c == '/':
            row += 1
            col = 0
        # if a digit is encountered then skip that many consecutive squares
        elif c.isdigit():
            col += int(c)
        # if character is found then place it on current row and column
        else:
            if c == 'P':
                board[row][col] = WHITE_PAWN
            elif c == 'R':
                board[row][col] = WHITE_ROOK
            elif c == 'N':
                board[row][col] = WHITE_KNIGHT
            elif c == 'B':
                board[row][col] = WHITE_BISHOP
            elif c == 'Q':
                board[row][col] = WHITE_QUEEN
            elif c == 'K':
                board[row][col] = WHITE_KING

            elif c == 'p':
                board[row][col] = BLACK_PAWN
            elif c == 'r':
                board[row][col] = BLACK_ROOK
            elif c == 'n':
                board[row][col] = BLACK_KNIGHT
            elif c == 'b':
                board[row][col] = BLACK_BISHOP
            elif c == 'q':
                board[row][col] = BLACK_QUEEN
            elif c == 'k':
                board[row][col] = BLACK_KING

            col += 1

    return board


data=pd.read_csv("data_test.csv")
data_dev=pd.read_csv("data_dev.csv")

tp=fromFEN(data["fen"][0])
tp

[[-2, 0, 0, 0, -6, -4, 0, -2],
 [-1, 0, 0, 0, 0, -1, -1, 0],
 [-4, 0, 0, 0, 0, -3, 0, -1],
 [0, -5, 0, -1, -1, 0, 0, 0],
 [3, 0, 0, 0, 1, 0, 0, 0],
 [0, 1, 0, 1, 0, 1, 0, 0],
 [1, 0, 1, 0, 0, 1, 0, 1],
 [2, 0, 4, 0, 0, 2, 6, 0]]

### getting features from current board representation

In [30]:
import subprocess

X=[]

def getFeatures(dataFrame):
    boards = []

    for data_str in dataFrame["fen"]:
        arr = fromFEN(data_str)

        for row in arr:
            boards.append(" ".join(map(str, row)))

    inp = f"{len(dataFrame)}\n" + "\n".join(boards) + "\n"

    result = subprocess.run(
        ["../feature"],
        input=inp,
        text=True,
        capture_output=True
    )

    return [
        list(map(float, line.split()))
        for line in result.stdout.strip().splitlines()
    ]

X=getFeatures(data)

X_train=pd.DataFrame(
    X,
    columns=["material","mobility","pawns","pressure","threat"]
)
X=getFeatures(data_dev)
X_dev=pd.DataFrame(
    X,
    columns=["material","mobility","pawns","pressure","threat"]
)
X_train.head()

,material,mobility,pawns,pressure,threat
0,-1038.0,-26.0,-20.0,0.0,0.0
1,208.0,41.0,15.0,-10.0,0.0
2,25.0,-20.0,0.0,0.0,0.0
3,-40.0,2.0,0.0,0.0,0.0
4,-648.0,-28.0,-15.0,10.0,0.0


In [62]:
scale=StandardScaler()
scale.fit(X_train)
X_train_scaled=scale.transform(X_train)
X_dev_scaled=scale.transform(X_dev)
Y_train=data[["score"]]
Y_dev=data_dev[["score"]]
model = Sequential([
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(64,activation='relu'),
    Dense(32,activation='relu'),
    Dense(20,activation='relu'),
    Dense(1)
])
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['accuracy']
)
model.fit(X_train_scaled,Y_train)
Y_pred=model.predict(X_dev_scaled)
err_dev = root_mean_squared_error(Y_dev,Y_pred)
Y_pred_train=model.predict(X_train_scaled)
err_train=root_mean_squared_error(Y_train,Y_pred_train)
print("Dev set Error: ",err_dev)
print("Train set Error: ",err_train)

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.0028 - loss: 86067.2812  
154/154 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  
155/155 ━━━━━━━━━━━━━━━━━━━━ 0s 577us/step
Dev set Error:  196.25064086914062
Train set Error:  190.26673889160156
